# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Pastikan data.csv tersedia di data/raw/."
    )

df = pl.read_csv(
    DATA_PATH,
    infer_schema_length=16000,
)

print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

Ukuran awal: 32,735 baris x 11 kolom


id,source,title,image,url,content,date,embedding,created_at,updated_at,summary
i64,str,str,str,str,str,str,str,str,str,str
83,"""tempo""","""Depo Plumpang Terbakar, Anggot…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Anggota Ko…","""2023-03-04 06:18:13+00""","""[-0.01590039,-0.034130897,0.00…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Anggota Komisi VII DPR RI Rofi…"
84,"""tempo""","""Jokowi Perintahkan Wapres Ma'r…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Presiden J…","""2023-03-04 06:04:38+00""","""[-0.017608976,-0.021786924,0.0…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Presiden Joko Widodo telah mem…"
85,"""tempo""","""HNW Mendukung Jamaah Umroh Fir…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Wakil Ketua MP…","""2023-03-04 06:18:04+00""","""[0.00841488,-0.023665192,0.006…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Wakil Ketua MPR RI Dr. H. M. H…"
86,"""tempo""","""Tim Dokkes Polri Telah Terima …","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Tim Kedokt…","""2023-03-04 06:44:10+00""","""[-0.012671886,-0.0039057182,0.…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Tim Kedokteran dan Kesehatan (…"
87,"""tempo""","""Bamsoet Ajak Komunitas Otomoti…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Ketua MPR RI s…","""2023-03-04 06:38:57+00""","""[-0.015486176,-0.0125719,-0.01…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Ketua MPR RI Bambang Soesatyo …"


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""33""","""63567""",32684,"""34081.74877042921""","""18044.06863841406""","""17745""","""35048""","""49828""",32735,0.00
"""source""","""VARCHAR""","""cnbcindonesia""","""tempo""",6,null,null,null,null,null,32735,0.00
"""title""","""VARCHAR""",""" Daftar Panglima Kostrad Dari …","""​​Tips Atasi Masalah Jerawat d…",30501,null,null,null,null,null,32735,0.00
"""image""","""VARCHAR""","""""","""https://statik.tempo.co/data/2…",35667,null,null,null,null,null,32735,0.00
"""url""","""VARCHAR""","""https://bisnis.tempo.co/read/1…","""https://www.suara.com/news/202…",35724,null,null,null,null,null,32735,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""date""","""VARCHAR""","""2018-08-10 05:11:58+00""","""2023-04-11 20:13:25+00""",33885,null,null,null,null,null,32735,0.00
"""embedding""","""VARCHAR""","""[-0.000100255646,-0.00392203,0…","""[9.975993e-05,-0.027434079,0.0…",33604,null,null,null,null,null,32735,0.00
"""created_at""","""VARCHAR""","""2023-03-03 16:08:03.144472""","""2023-04-11 20:30:13.315332""",3026,null,null,null,null,null,32735,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""summary""",10,0.03


## 3. Penanganan Missing Values

Strategi yang dipakai disesuaikan dengan peran kolom. Baris dengan `summary` null dihapus karena ringkasan diperlukan untuk membuat fitur panjang teks dan analisis berikutnya. Null pada `content` dan `image` dipertahankan karena keduanya bukan syarat wajib untuk setiap artikel; kondisi tersebut dicatat sebagai bagian dari kualitas data.

In [4]:
null_sebelum = int(df.null_count().sum_horizontal().sum())
baris_sebelum_missing = len(df)
df_missing_handled = df.drop_nulls(subset=['summary'])
baris_dihapus_missing = baris_sebelum_missing - len(df_missing_handled)

print(f'Null sebelum: {null_sebelum:,}')
print(f'Baris dihapus karena summary null: {baris_dihapus_missing:,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')
print('Null content/image dipertahankan karena bukan kolom wajib.')

Null sebelum: 10
Baris dihapus karena summary null: 10
Null sesudah: 0
Null content/image dipertahankan karena bukan kolom wajib.


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [5]:
df_missing_handled = df_missing_handled.with_columns(
    pl.col('title').str.len_chars().alias('panjang_judul'),
    pl.col('summary').str.split(' ').list.len().alias('jumlah_kata_summary'),
)

KOLOM_NUMERIK = 'jumlah_kata_summary'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
jumlah_outlier = df_missing_handled.filter(
    (pl.col(KOLOM_NUMERIK) < batas_bawah)
    | (pl.col(KOLOM_NUMERIK) > batas_atas)
).height
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')
print(f'Jumlah outlier sebelum penanganan: {jumlah_outlier:,}')

df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)
print('Strategi outlier: winsorization, yaitu membatasi nilai ke rentang IQR tanpa menghapus baris.')

Q1=51.0, Q3=72.0, IQR=21.0
Batas bawah: 19.5, Batas atas: 103.5
Jumlah outlier sebelum penanganan: 154
Strategi outlier: winsorization, yaitu membatasi nilai ke rentang IQR tanpa menghapus baris.


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [6]:
KOLOM_KUNCI = 'url'

total_baris = len(df_outlier_handled)
id_unik = df_outlier_handled['id'].n_unique()
url_unik = df_outlier_handled['url'].n_unique()
baris_unik = url_unik
duplikat_id = total_baris - id_unik
duplikat_url = total_baris - url_unik
print(f'Total baris: {total_baris:,}')
print(f'ID unik: {id_unik:,}; duplikat ID: {duplikat_id:,}')
print(f'URL unik: {url_unik:,}; duplikat URL: {duplikat_url:,}')
print('Kunci deduplikasi: url, karena satu URL merepresentasikan satu artikel.')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

Total baris: 32,725
ID unik: 32,725; duplikat ID: 0
URL unik: 32,725; duplikat URL: 0
Kunci deduplikasi: url, karena satu URL merepresentasikan satu artikel.


## 6. Validasi Data Dasar

Validasi memeriksa kunci artikel, kolom wajib, sumber berita, format URL, serta waktu publikasi. Setiap aturan menghasilkan jumlah pelanggaran dan status `LULUS` atau `GAGAL`.

In [7]:
connection.register('dataset_bersih', df_dedup)

validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(*) - COUNT(DISTINCT id) AS id_duplikat,
        COUNT(*) - COUNT(DISTINCT url) AS url_duplikat,
        SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS id_kosong,
        COUNT(DISTINCT source) AS jumlah_sumber,
        SUM(CASE WHEN source IS NULL OR trim(source) = '' THEN 1 ELSE 0 END) AS source_kosong,
        SUM(CASE WHEN source NOT IN ('tempo', 'cnbcindonesia', 'cnnindonesia', 'kumparan', 'okezone', 'jawapos', 'suara') THEN 1 ELSE 0 END) AS source_tidak_dikenal,
        SUM(CASE WHEN url IS NULL OR trim(url) = '' THEN 1 ELSE 0 END) AS url_kosong,
        SUM(CASE WHEN url IS NOT NULL AND NOT regexp_matches(url, '^https?://') THEN 1 ELSE 0 END) AS url_tidak_valid,
        SUM(CASE WHEN trim(title) = '' THEN 1 ELSE 0 END) AS judul_kosong,
        SUM(CASE WHEN try_cast(date AS TIMESTAMPTZ) IS NULL THEN 1 ELSE 0 END) AS tanggal_tidak_valid,
        SUM(CASE WHEN try_cast(date AS TIMESTAMPTZ) > now() THEN 1 ELSE 0 END) AS tanggal_masa_depan
    FROM dataset_bersih
""").pl()
pelanggaran = int(validasi.select(
    pl.exclude(['total_baris', 'jumlah_sumber'])
).sum_horizontal().item())
validasi = validasi.with_columns(
    pl.when(pl.lit(pelanggaran) == 0)
    .then(pl.lit('LULUS'))
    .otherwise(pl.lit('PERLU PERBAIKAN'))
    .alias('status_validasi')
)
validasi

total_baris,id_duplikat,url_duplikat,id_kosong,jumlah_sumber,source_kosong,source_tidak_dikenal,url_kosong,url_tidak_valid,judul_kosong,tanggal_tidak_valid,tanggal_masa_depan,status_validasi
i64,i64,i64,"decimal[38,0]",i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",str
32725,0,0,0,7,0,0,0,0,0,0,0,"""LULUS"""


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 32,735
Baris sesudah cleaning: 32,725
Data bersih tersimpan di: C:\Users\maull\Downloads\Kuliah\SMT 7\Analisis Big Data\Tugas1\tugas1-abd-2026\data\processed\dataset_bersih.parquet


## Catatan Milestone 2

Ringkasan cleaning: baris dengan `summary` null dihapus karena kolom tersebut dipakai untuk fitur teks; null `content` dan `image` dipertahankan karena bukan kolom wajib. Outlier `jumlah_kata_summary` ditangani dengan winsorization berbasis IQR agar baris artikel tidak hilang. Duplikasi diperiksa pada `id` dan `url`, lalu deduplikasi dilakukan berdasarkan `url` dengan mempertahankan baris pertama. Validasi akhir memeriksa ID, URL, sumber, judul, dan tanggal sebelum hasil disimpan sebagai Parquet.